In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import yaml
import torch
import pandas as pd
from pathlib import Path
import torch
import torch.nn.functional as F

In [14]:
def build_token_ranges(config_modalities):
    token_ranges = {
        "CLS": slice(0, 1)
    }
    
    current_idx = 1

    for mod in config_modalities:
        name = mod["name"].upper()
        length = mod.get("max_pad", 1)

        start = current_idx
        end = current_idx + length

        token_ranges[name] = slice(start, end)

        current_idx = end

    return token_ranges

In [15]:
def check_img_similarity(pids, modality):
    # Load all .npz feature files for a patient/modality and compute pairwise cosine similarity.
    if modality == 'WSI':
        feat_dir = Path(f"../lung/features/avg-wsi")
    elif modality=='CT':
        feat_dir = Path(f"../lung/features/CT")
    files = []
    for pid in pids:
        files.extend(sorted(feat_dir.glob(f"{pid}*.*")))

    if len(files) == 0:
        print(
            f"No files found for {pid} in {feat_dir}"
        )
        return None

    print(f"Found {len(files)} files")
    feats = []
    names = []

    file_ext = files[0].suffix.lower().lstrip(".")
    
    for f in files:
        
        if file_ext == 'npz': 
            data = np.load(f)
            arr_name = list(data.keys())[0]
            feat = data[arr_name]
        else: # .pt
            data = torch.load(f"{f}")
            feat = data.cpu()  #torch.Size([1, 512, 1, 1, 1])

        # flatten in case features are spatial
        feat = feat.reshape(-1)

        feats.append(feat)
        stem = f.stem
        parts = stem.split("-")
        names.append(parts[0]+'-'+parts[1]+'_'+parts[-1])

    feats = np.stack(feats)

    # =====================================================
    # COSINE SIMILARITY
    # =====================================================

    x = torch.tensor(feats, dtype=torch.float32)
    x = F.normalize(x, dim=-1)

    sim = x @ x.T
    sim = sim.cpu().numpy()

    # =====================================================
    # PRINT
    # =====================================================

    print("\nCosine similarity matrix:\n")
    print(np.round(sim, 3))

    # =====================================================
    # PLOT
    # =====================================================

    plt.figure(figsize=(12, 8))

    sns.heatmap(
        sim,
        annot=True,
        fmt=".3f",
        xticklabels=names,
        yticklabels=names,
        square=True,
        cmap="viridis",
        vmin=-1,
        vmax=1,
    )

    plt.title(
        f"{pid} | {modality} feature similarity"
    )

    plt.tight_layout()
    plt.show()

    return sim

In [16]:
def plot_attn_matrix(all_patient_seq, token_ranges, pid, layer_idx=-1, plot=True):
    """
    Check CLS attention scores for a certain patient ID (both per token and averaged per modality)
    """
    diag = pd.read_csv('../lung/features/pretty/CPTAC+TCGA/diagnosis-encoded.csv').fillna(-1)
    dem = pd.read_csv('../lung/features/pretty/CPTAC+TCGA/demographics-encoded.csv').fillna(-1)
    clin_df = pd.merge(
        diag,
        dem,
        on='cases.submitter_id',
        how='inner'   # or 'outer' depending on whether you want all samples
    )
    pid_clin = clin_df[clin_df['cases.submitter_id']==pid]

    patient_seq = all_patient_seq[pid]
    token_names = patient_seq["token_names"]

    # keep only non-PAD tokens
    keep_mask = [not name.endswith("_PAD") for name in token_names]

    # filter token names
    token_names = [name for name, keep in zip(token_names, keep_mask) if keep]
    
    cls_attn = patient_seq['cls_attn']
    cls_attn_mean_heads = patient_seq['cls_attn_mean_heads']
    cls_rollout = patient_seq['cls_rollout']
    
    attn_layer = cls_attn[layer_idx]
    cls_attn_mean_heads = cls_attn_mean_heads[layer_idx]
    # filter attention
    attn_layer = attn_layer[:, keep_mask]    
    cls_attn_mean_heads = cls_attn_mean_heads[keep_mask]
    cls_rollout = cls_rollout[keep_mask]
    
    pred = patient_seq['pred']
    label = patient_seq['label']
    prob = patient_seq['prob']
    #print(f"Attention layer{layer_idx}:\n",attn_layer)
    num_heads = attn_layer.shape[0]
    
    num_heads = attn_layer.shape[0]

    rollout_np = cls_rollout.reshape(-1)
    #top3_indices = np.argsort(rollout_np)[-4:][::-1] # CLS + top3
    #top3_values = rollout_np[top3_indices]
    #top3_features = [token_names[i] for i in top3_indices if token_names[i]!='CLS']
    #print("top3 feats", top3_features)

    #pid_clin = pid_clin[top3_features]
    #missing_feats = [
    #    f for f in top3_features
    #    if pid_clin[f].iloc[0] == -1
    #]
    missing_feats=None

    if plot: 
        rollout = cls_rollout.reshape(1, -1)
      
        print("rollout",rollout)
        annot = np.where(
            rollout == 0,
            "–",                          # or "NA" or "∅"
            np.round(rollout, 3).astype(str)
        )
        
        fig, axes = plt.subplots(
            nrows=3,
            figsize=(30, 4),
        )
        fig.subplots_adjust(hspace=1)
    
        sns.heatmap(
            cls_attn_mean_heads.reshape(1, -1),
            cmap="viridis",
            cbar=True,
            #xticklabels=token_names,
            yticklabels=["AVG Heads"],
            annot=cls_attn_mean_heads.reshape(1, -1),
            fmt=".3f",
            ax=axes[0],
        )
    
        axes[0].set_title(
            f"{pid} | Layer {layer_idx} | "
            f"Pred: {int(pred)} | Label: {int(label)} | Prob: {prob*100:.2f}%"
        )
    
    
        
        sns.heatmap(
            attn_layer,
            cmap="viridis",
            cbar=True,
            #xticklabels=token_names,
            yticklabels=[f"H{i}" for i in range(num_heads)],
            annot=True,
            fmt=".3f",
            ax=axes[1],
        )
    
        axes[1].set_title("Per-Head CLS Attention")
    
    
        sns.heatmap(
            rollout,
            cmap="viridis",
            cbar=True,
            xticklabels=token_names,
            yticklabels=["rollout"],
            annot=annot ,
            fmt="",
            ax=axes[2],
        )
    
        axes[2].set_title("Rollout")
    
        plt.show()

    if missing_feats:
        print(f"{pid} Missing Feats TOP3 {pid_clin.to_string()}")
        return missing_feats



In [17]:
def find_patient_sequence(results, pid):
    """
    Returns the patient_seq and fold_idx for a given patient ID
    """
    for i, fold in enumerate(results["folds"]):
        test = unwrap(fold["test_results"])
        patient_seqs = test["patient_sequences"]
        if pid in patient_seqs:
            return patient_seqs, i
            
    print(f"Patient {pid} not found in any fold")
    return None, None
    
def to_float(x):
    if isinstance(x, np.ndarray):
        return float(x)
    return float(x)
def unwrap(test):
    if isinstance(test, tuple):
        return test[0]
    return test
def collect_test_fold_metrics(results):
    balaccs = []
    aucs = []
    for fold in results["folds"]: 
        test = unwrap(fold["test_results"])
        balaccs.append(to_float(test["balacc"]))
        
        aucs.append(to_float(test["roc_auc"]))

    best_idx = int(np.argmax(aucs))

    best_fold = results["folds"][best_idx]
    best_test = unwrap(best_fold["test_results"])
    #print(best_test)
    print(
        f"Best fold: {best_idx}, "
        f"Decision Threshold: {best_test['decision_threshold']}, "
        f"BalAcc: {to_float(best_test['balacc']) * 100:.2f}%, "
        f"ROCAUC: {to_float(best_test['roc_auc']) * 100:.2f}%"
    )
    print("-- 5 Fold Average:")
    print("Balanced Accuracy: "
      f"{np.mean(balaccs)*100:.2f} ± {np.std(balaccs)*100:.2f}%")

    print("ROC AUC: "
          f"{np.mean(aucs)*100:.2f} ± {np.std(aucs)*100:.2f}%")

    return best_test

In [18]:
import os
pids = []
pid_dict = {}
for f in os.listdir('../../data/lung/slide_features_titan'):
    parts=f.split(".")

    if parts[0].startswith('TCGA'):
        p = parts[0].split("-")
        pid=p[0]+'-'+p[1]+'-'+p[2]
        if pid not in pid_dict.keys():
            pid_dict[pid]=1
        else:
            pid_dict[pid]+=1
    else:
        p = parts[0].split("-")
        pid=p[0]+'-'+p[1]

    
        
    pids.append(pid)
# TCGA-49-4488    TCGA-33-4547

In [20]:
# load config file
with open('config.yaml', "r") as f:
    cfg = yaml.safe_load(f)

modalities = cfg["modalities"]
token_ranges = build_token_ranges(modalities)
clinical_data = pd.read_csv('../lung/features/TCGA-encoded/new-demographics-encoded.csv')
all_patient_ids = clinical_data['cases.submitter_id']
# load experiment data
exp_dir = '0708_0050-unimodal-demographics-diagnosis'
pid= 'C3N-02434'

data = torch.load(f"experiments/{exp_dir}/cv_results.pt")
best_test_fold = collect_test_fold_metrics(data)
# SINGLE PID
if pid is not None:
    all_pid_sequence, fold_idx = find_patient_sequence(data, pid) # find pid fold
    plot_attn_matrix(all_pid_sequence,token_ranges,pid=pid, layer_idx=-1)


# ALL PIDS
else:
    top3=[]
    # iterate all folds
    for fold_idx, fold_data in enumerate(data):

        for pid in all_patient_ids:
            try:
                all_pid_sequence, _ = find_patient_sequence(data, pid)
                aux = plot_attn_matrix(all_pid_sequence,token_ranges,pid=pid,plot=False)
                if aux is not None:
                    top3.append(aux)

            except Exception as e:
                print(f"Failed for {pid}: {e}")

Best fold: 2, Decision Threshold: 0.5011138916015625, BalAcc: 61.94%, ROCAUC: 62.18%
-- 5 Fold Average:
Balanced Accuracy: 56.74 ± 4.39%
ROC AUC: 53.88 ± 5.86%
Patient C3N-02434 not found in any fold


/tmp/ipykernel_2007/2088450213.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f"experiments/{exp_dir}/cv_results.pt")


TypeError: 'NoneType' object is not subscriptable